In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm
import re
import sys
from pathlib import Path
import glob
import subprocess

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11
Rasterio version: 1.4.3


In [2]:
# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing_local import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities_local import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


In [3]:
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'
local_path = "local-files"

###

EVENT_NAME = "202409_TropicalStorm_Francine"
product = "sentinel1"

In [4]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

keys = [x.replace(f"drcs_activations/{EVENT_NAME}/{product}/", "") for x in get_all_s3_keys(s3_client, BUCKET, f"drcs_activations/{EVENT_NAME}/{product}", ".tif")] if s3_client else []

keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files


['S1A_IW_20240906T000209_DVR_RTC20_G_gpuned_3F65_WM.tif',
 'S1A_IW_20240906T000209_DVR_RTC20_G_gpuned_3F65_rgb.tif',
 'S1A_IW_20240906T000234_DVR_RTC20_G_gpuned_8248_WM.tif',
 'S1A_IW_20240906T000234_DVR_RTC20_G_gpuned_8248_rgb.tif',
 'S1A_IW_20240906T000259_DVR_RTC20_G_gpuned_AAF9_WM.tif',
 'S1A_IW_20240906T000259_DVR_RTC20_G_gpuned_AAF9_rgb.tif',
 'S1A_IW_20240906T000324_DVR_RTC20_G_gpuned_BDBA_WM.tif',
 'S1A_IW_20240906T000324_DVR_RTC20_G_gpuned_BDBA_rgb.tif',
 'S1A_IW_20240906T000348_DVR_RTC20_G_gpuned_1028_WM.tif',
 'S1A_IW_20240906T000348_DVR_RTC20_G_gpuned_1028_rgb.tif',
 'S1A_IW_20240911T001010_DVR_RTC20_G_gpuned_5E0B_WM.tif',
 'S1A_IW_20240911T001010_DVR_RTC20_G_gpuned_5E0B_rgb.tif',
 'S1A_IW_20240911T001036_DVR_RTC20_G_gpuned_C0DB_WM.tif',
 'S1A_IW_20240911T001036_DVR_RTC20_G_gpuned_C0DB_rgb.tif',
 'S1A_IW_20240911T001100_DVR_RTC20_G_gpuned_8BE9_WM.tif',
 'S1A_IW_20240911T001100_DVR_RTC20_G_gpuned_8BE9_rgb.tif',
 'S1A_IW_20240911T001125_DVR_RTC20_G_gpuned_6C28_WM.tif',
 'S1A_

In [5]:
# Download the desired files to a local directory with a known path
local_file_dir = os.path.abspath(f"./{local_path}")
if not os.path.exists(local_file_dir):
    os.mkdir(local_file_dir)
for key in keys:
    subprocess.run([
        "aws",
        "s3",
        "cp",
        f"s3://nasa-disasters/drcs_activations/{EVENT_NAME}/{product}/{key}",
        local_file_dir], check = True)

download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/sentinel1/S1A_IW_20240906T000209_DVR_RTC20_G_gpuned_3F65_WM.tif to local-files/S1A_IW_20240906T000209_DVR_RTC20_G_gpuned_3F65_WM.tif
download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/sentinel1/S1A_IW_20240906T000209_DVR_RTC20_G_gpuned_3F65_rgb.tif to local-files/S1A_IW_20240906T000209_DVR_RTC20_G_gpuned_3F65_rgb.tif
download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/sentinel1/S1A_IW_20240906T000234_DVR_RTC20_G_gpuned_8248_WM.tif to local-files/S1A_IW_20240906T000234_DVR_RTC20_G_gpuned_8248_WM.tif
download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/sentinel1/S1A_IW_20240906T000234_DVR_RTC20_G_gpuned_8248_rgb.tif to local-files/S1A_IW_20240906T000234_DVR_RTC20_G_gpuned_8248_rgb.tif
download: s3://nasa-disasters/drcs_activations/202409_TropicalStorm_Francine/sentinel1/S1A_IW_20240906T000259_DVR_RTC20_G_gpuned_AAF9_WM.tif to local-files/

In [6]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()


# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

In [7]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [8]:
def make_regex_dict(keys, regexes, products):
    ret = {}
    for i in range(len(products)):
        matches = []
        for key in keys:
            filename = key.split("/")[-1]
            match = re.search(regexes[i], filename)
            if match is not None:
                matches.append(key)
        if matches != []:
            ret[products[i]] = matches
    return ret

In [9]:
def create_cog_filename(filename, event):
    if re.search(r".*_rgb.tif", filename) is not None:
        sname = filename.split("/")[-1].replace(".tif", "").split("_")
        date = datetime.strptime(sname[2], "%Y%m%dT%H%M%S")
        new_dt_format = date.strftime("%Y-%m-%dT%H:%M:%SZ")
        cog_filename = f"{event}_{sname[0]}_{sname[1]}_{sname[3]}_{sname[4]}_{sname[5]}_{sname[6]}_{sname[7]}_{sname[8]}_{new_dt_format}.tif"

    elif re.search(r".*_WM.tif", filename) is not None:
        sname = filename.split("/")[-1].replace(".tif", "").split("_")
        date = datetime.strptime(sname[2], "%Y%m%dT%H%M%S")
        new_dt_format = date.strftime("%Y-%m-%dT%H:%M:%SZ")
        cog_filename = f"{event}_{sname[0]}_{sname[1]}_{sname[3]}_{sname[4]}_{sname[5]}_{sname[6]}_{sname[7]}_{sname[8]}_{new_dt_format}.tif"

    else:
        print(f"{filename} not caught by regexes!")
        return None
    
    return cog_filename

In [10]:
local_keys = [x for x in glob.glob(f"{local_path}/*") if x.endswith(".tif")]

local_keys

['local-files/S1A_IW_20240906T000209_DVR_RTC20_G_gpuned_3F65_WM.tif',
 'local-files/S1A_IW_20240906T000209_DVR_RTC20_G_gpuned_3F65_rgb.tif',
 'local-files/S1A_IW_20240906T000234_DVR_RTC20_G_gpuned_8248_WM.tif',
 'local-files/S1A_IW_20240906T000234_DVR_RTC20_G_gpuned_8248_rgb.tif',
 'local-files/S1A_IW_20240906T000259_DVR_RTC20_G_gpuned_AAF9_WM.tif',
 'local-files/S1A_IW_20240906T000259_DVR_RTC20_G_gpuned_AAF9_rgb.tif',
 'local-files/S1A_IW_20240906T000324_DVR_RTC20_G_gpuned_BDBA_WM.tif',
 'local-files/S1A_IW_20240906T000324_DVR_RTC20_G_gpuned_BDBA_rgb.tif',
 'local-files/S1A_IW_20240906T000348_DVR_RTC20_G_gpuned_1028_WM.tif',
 'local-files/S1A_IW_20240906T000348_DVR_RTC20_G_gpuned_1028_rgb.tif',
 'local-files/S1A_IW_20240911T001010_DVR_RTC20_G_gpuned_5E0B_WM.tif',
 'local-files/S1A_IW_20240911T001010_DVR_RTC20_G_gpuned_5E0B_rgb.tif',
 'local-files/S1A_IW_20240911T001036_DVR_RTC20_G_gpuned_C0DB_WM.tif',
 'local-files/S1A_IW_20240911T001036_DVR_RTC20_G_gpuned_C0DB_rgb.tif',
 'local-files

In [11]:
reg_keys = make_regex_dict(local_keys, [r".*_rgb.tif", r".*_WM.tif"], ["RGB/subdaily", "HydroSAR_WM"])

In [12]:
print(reg_keys)
for k, v in reg_keys.items():
    for filename in v:
        print(create_cog_filename(filename, EVENT_NAME))

{'RGB/subdaily': ['local-files/S1A_IW_20240906T000209_DVR_RTC20_G_gpuned_3F65_rgb.tif', 'local-files/S1A_IW_20240906T000234_DVR_RTC20_G_gpuned_8248_rgb.tif', 'local-files/S1A_IW_20240906T000259_DVR_RTC20_G_gpuned_AAF9_rgb.tif', 'local-files/S1A_IW_20240906T000324_DVR_RTC20_G_gpuned_BDBA_rgb.tif', 'local-files/S1A_IW_20240906T000348_DVR_RTC20_G_gpuned_1028_rgb.tif', 'local-files/S1A_IW_20240911T001010_DVR_RTC20_G_gpuned_5E0B_rgb.tif', 'local-files/S1A_IW_20240911T001036_DVR_RTC20_G_gpuned_C0DB_rgb.tif', 'local-files/S1A_IW_20240911T001100_DVR_RTC20_G_gpuned_8BE9_rgb.tif', 'local-files/S1A_IW_20240911T001125_DVR_RTC20_G_gpuned_6C28_rgb.tif'], 'HydroSAR_WM': ['local-files/S1A_IW_20240906T000209_DVR_RTC20_G_gpuned_3F65_WM.tif', 'local-files/S1A_IW_20240906T000234_DVR_RTC20_G_gpuned_8248_WM.tif', 'local-files/S1A_IW_20240906T000259_DVR_RTC20_G_gpuned_AAF9_WM.tif', 'local-files/S1A_IW_20240906T000324_DVR_RTC20_G_gpuned_BDBA_WM.tif', 'local-files/S1A_IW_20240906T000348_DVR_RTC20_G_gpuned_1028

In [13]:
def simple_process_files(file_list, rename_func, target_dir, event):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    print("Testing filenams:")
    for filename in file_list:
        print(f"  {rename_func(filename, event)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        #"raw_data_bucket": BUCKET,
        #"raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{event}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(local_download_path, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            local_download_path, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=file_list,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=event,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

In [14]:
if not os.path.exists(os.path.abspath("./output")):
    os.mkdir(os.path.abspath("./output"))
if not os.path.exists(os.path.abspath("./reproj")):
    os.mkdir(os.path.abspath("./reproj"))
for k, v in reg_keys.items():
    results = simple_process_files(file_list = v, rename_func = create_cog_filename, target_dir = f"Sentinel-1/{k}", event = EVENT_NAME)

Testing filenams:
  202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_3F65_rgb_2024-09-06T00:02:09Z.tif
  202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_8248_rgb_2024-09-06T00:02:34Z.tif
  202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_AAF9_rgb_2024-09-06T00:02:59Z.tif
  202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_BDBA_rgb_2024-09-06T00:03:24Z.tif
  202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_1028_rgb_2024-09-06T00:03:48Z.tif
  202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_5E0B_rgb_2024-09-11T00:10:10Z.tif
  202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_C0DB_rgb_2024-09-11T00:10:36Z.tif
  202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_8BE9_rgb_2024-09-11T00:11:00Z.tif
  202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_6C28_rgb_2024-09-11T00:11:25Z.tif
Configuration loaded:
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Sentinel-1/RGB/subdaily

🌊 Processing Files (Chunked)
✅ Local 

Band 1:  51%|█████     | 76/150 [00:03<00:03, 22.15chunks/s]


   [MEMORY] High usage: 608.0 MB, forcing cleanup...


Band 1:  57%|█████▋    | 86/150 [00:03<00:03, 20.39chunks/s]


   [MEMORY] High usage: 647.9 MB, forcing cleanup...


...

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpxmw0dwqa.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/HydroSAR_WM/202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_8BE9_WM_2024-09-11T00:11:00Z.tif
   [MEMORY] Final: 1078.8 MB (Change: -41.7 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_8BE9_WM_2024-09-11T00:11:00Z.tif

[9/9] Processing: local-files/S1A_IW_20240911T001125_DVR_RTC20_G_gpuned_6C28_WM.tif
   Output filename: 202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_6C28_WM_2024-09-11T00:11:25Z.tif
   [CACHE HIT] Using local file: local-files/S1A_IW_20240911T001125_DVR_RTC20_G_gpuned_6C28_WM.tif
   [MEMORY] Initial: 1078.8 MB
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 

   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=255, center sample non-zero=999969/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...
   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Reading input: /tmp/tmp3lcykmbv_temp.tif

Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmprwjwhg5_.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Sentinel-1/HydroSAR_WM/202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_6C28_WM_2024-09-11T00:11:25Z.tif
   [MEMORY] Final: 1130.3 MB (Change: +51.5 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202409_TropicalStorm_Francine_S1A_IW_DVR_RTC20_G_gpuned_6C28_WM_2024-09-11T00:11:25Z.tif

✅ Batch processing complete: 9 files processed
📁 COGs saved locally to: output/202409_TropicalStorm_Francine

📊 BATCH PROCESSING SUMMARY
Total files processed: 9
Successful: 9
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-29T22:56:39.812853


In [15]:
subprocess.run(["rm", "-r", f"{local_path}"], check = True)
subprocess.run(["rm", "-r", os.path.abspath("./output")], check = True)
subprocess.run(["rm", "-r", os.path.abspath("./reproj")], check = True)

CompletedProcess(args=['rm', '-r', '/home/jovyan/conversion_scripts/convert-files-and-move/2024/reproj'], returncode=0)